In [ ]:
#1 패키지 불러오기_25.4.1
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
from pop import Pilot, Camera
import time

In [ ]:
#2 모델 정의
class AlexNet(nn.Module):
    def __init__(self):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2)),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2)),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )
        self.avgpool = nn.AdaptiveAvgPool2d(output_size=(6, 6))
        self.classifier = nn.Sequential(
            nn.Linear(9216, 4096),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 265),
            nn.ReLU(),
            nn.Linear(265, 2)  # 2개의 클래스로 분류 (block, free)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
#3 학습된 모델 가져오기
class MyTransferLearningModel(nn.Module):
    def __init__(self, model):
        super(MyTransferLearningModel, self).__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)

In [ ]:
#4. 모델 불러오기
model = MyTransferLearningModel(AlexNet())

# 학습모델로 이름 바꾸기 .pht 파일만 가능
model.load_state_dict(torch.load('best_model_04192.pth'))
#model.load_state_dict(torch.load('/home/soda/Project/python/notebook/best_model.pth')) 승호도움
model.eval()  # 평가 모드로 설정

In [ ]:
#5. 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # AlexNet 입력 크기
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
#6. 모듈 연결 작업
autocar = Pilot.AutoCar()         #자율자행차량 객체 생성
camera = Camera()                #카메라 초기화

In [ ]:
#7. 프레임을 PIL 이미지로 변환
def detect_obstacle(frame):
    # 프레임을 PIL 이미지로 변환
    pil_image = Image.fromarray(frame)

    # 전처리
    input_tensor = transform(pil_image).unsqueeze(0)

    # 모델 예측
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        prob, pred = torch.max(probs, 1)

    print(f"예측: {pred.item()}, 확률: {prob.item():.2f}")

    if prob.item() < 0.8:
        print("확률 낮음 → 무시하고 전진")
        return 1  # 일시적으로 free로 판단
    return pred.item()

    return pred.item()


In [ ]:
#8. 자동차 제어 함수
def control_car(prediction):
    if prediction == 0:  # 장애물 (block)
        autocar.stop()
        print("장애물 발견! 멈춤!")
    else:  # 자유 (free)
        print("길이 비었습니다! 주행 시작!")
        autocar.forward(30)         #속도 30

In [ ]:
#9. 실제 동작
try:
    while True:
        frame = camera.value  # 실시간 영상 프레임 받기
        prediction = detect_obstacle(frame)  # 예측 수행
        control_car(prediction)              # 예측값으로 제어
        time.sleep(1)                        # 주기적 실행
except KeyboardInterrupt:
    autocar.stop()
    print("종료됨")

Image.fromarray(frame).save("test_frame.jpg")

In [ ]:
autocar.stop()